# Data Types and Formats in EDAM

**The ontology of data analysis and management**

Links:

* https://edamontology.org/page
* https://edamontologydocs.readthedocs.io

## EDAM Overview - focus on data type

EDAM includes 4 main sections of concepts (sub-ontologies):

* Topic - “A category denoting a rather broad domain or field of interest, of study, application, work, data, or technology. Topics have no clearly defined borders between each other.”
* Operation - “A function that processes a set of inputs and results in a set of outputs, or associates arguments (inputs) with values (outputs).”
* **Data** - “Information, represented in an information artefact (data record) that is 'understandable' by dedicated computational tools that can use the data as input or produce it as output.”
* **Format** - “A defined way or layout of representing and structuring data in a computer file, blob, string, message, or elsewhere.”

![EDAM relations diagram](https://edamontology.org/EDAMrelations.png)

## Exploration Venues for EDAN

There are some potential venues to explore, which use EDAN as resource containing linked data on (data) *formats* and *data types*, namely:

* Data classification classes
* links between  Data classes and Format classes
  * properties: edam:is_format_of, edam:has_format
  * example:  edam:format_3547 edam:is_format_of edam:data_2968 <s>(Note: this relation is not expressed in [EDAM_1.25.owl](https://edamontology.org/EDAM_1.25.owl) but on a  [EDAM format->data mappings spreadsheet]() )</s>

## Major issues

**Incompleteness/Narrow scope:**

It seems that only the perspective of one community, or group of users is expressed in EDAN.
Take for instance the statement *edam:format_3915 ("Zarr") is format of edam:format_3915 ("Sequence tag profile") and edam:data_3112 ("Gene expression matrix")*. Such statement leaves out the use of ZARR for N-Array data (Note: N-Array data or similar is NOT defined in EDAM)  


## EDAM: links between Data classes and Format classes

![Detail of EDAM Data class children](../media/EDAM-data.png)


### complex is_format_of OWL structures

is_format_of relationship, in EDAM are expressed as owl:Restrictions, as expressed below for edam:format_1475 , which state: *the subject format_1475 on property (owl:onProperty) is_format_of, is subject to a restriction(owl:Restriction), applicable (owl:someValuesFrom) to entity data_0883*. In other words *format_1475 is_format_of data_0883*


```xml
    <owl:Class rdf:about="http://edamontology.org/format_1475">
        <rdfs:subClassOf rdf:resource="http://edamontology.org/format_2033"/>
        <rdfs:subClassOf>
            <owl:Restriction>
                <owl:onProperty rdf:resource="http://edamontology.org/is_format_of"/>
                <owl:someValuesFrom rdf:resource="http://edamontology.org/data_0883"/>
            </owl:Restriction>
        </rdfs:subClassOf>
        <rdfs:subClassOf>
            <owl:Restriction>
                <owl:onProperty rdf:resource="http://edamontology.org/is_format_of"/>
                <owl:someValuesFrom rdf:resource="http://edamontology.org/data_3870"/>
            </owl:Restriction>
        </rdfs:subClassOf>
        <created_in>beta12orEarlier</created_in>
        <oboInOwl:hasDefinition>Format of an entry (or part of an entry) from the PDB database.</oboInOwl:hasDefinition>
        <oboInOwl:hasExactSynonym>PDB entry format</oboInOwl:hasExactSynonym>
        <oboInOwl:inSubset rdf:resource="http://purl.obolibrary.org/obo/edam#edam"/>
        <oboInOwl:inSubset rdf:resource="http://purl.obolibrary.org/obo/edam#formats"/>
        <rdfs:label>PDB database entry format</rdfs:label>
    </owl:Class>
```


In [23]:
# boiler plate functions
# imports SPARQL prefixes and functions defs

from rdflib import Graph
from prettytable import PrettyTable

prefixes = '''    
PREFIX edam: <http://edamontology.org/>
PREFIX oboinowl: <http://www.geneontology.org/formats/oboInOwl#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
'''    

def sparql_files_query(query, format, filespath):
    g = Graph()
    for filepath in filespath:
        g.parse(filepath, format=format)  # can also use "ttl" for Turtle
    query = prefixes + query
    results = g.query(query)
    return results


def sparql_resutls_to_prettytable(results):
    table = PrettyTable()
    table.field_names = results.vars
    for row in results:
        table.add_row(row)
    return table

In [22]:
# Output: relations between formats and data in EDAM 
# where the format is a subclass of edam:format_1915 (label:format)

q_edam_format_data_relations = '''
SELECT ?format ?format_label ?data ?data_label
WHERE {

        ?format rdfs:subClassOf+ edam:format_1915 ;
            rdfs:label ?format_label .
        ?format rdfs:subClassOf ?restriction .
        ?restriction owl:onProperty edam:is_format_of ;
                     owl:someValuesFrom ?data .
        ?data rdfs:label ?data_label .
}
ORDER BY ?format_label
'''

r_edam_format_data_relations = sparql_files_query(query=q_edam_format_data_relations,
                                           filespath=['https://edamontology.org/EDAM_1.25.owl'],
                                           format='xml'
                                           )
print(sparql_resutls_to_prettytable(results=r_edam_format_data_relations))


+-------------------------------------+-------------------------------------------------------+-----------------------------------+-------------------------------------------+
|                format               |                      format_label                     |                data               |                 data_label                |
+-------------------------------------+-------------------------------------------------------+-----------------------------------+-------------------------------------------+
| http://edamontology.org/format_2064 |              3D-1D scoring matrix format              | http://edamontology.org/data_1499 |            3D-1D scoring matrix           |
| http://edamontology.org/format_3708 |                      ABCD format                      | http://edamontology.org/data_3707 |             Biodiversity data             |
| http://edamontology.org/format_1921 |                    Alignment format                   | http://edamontology.org/

In [21]:
# Output: the data types associated with the format edam:format_3915 (label: Zarr) in EDAM

q_edam_format_3915_data_relations = '''
SELECT ?format ?format_label ?data ?data_label
WHERE {
    BIND ( edam:format_3915 AS ?format )
        ?format rdfs:subClassOf ?restriction ;
                    rdfs:label ?format_label .
        ?restriction owl:onProperty edam:is_format_of ;
                        owl:someValuesFrom ?data .
        ?data rdfs:label ?data_label .
}
ORDER BY ?format_label
'''

r_edam_format_3915_data_relations = sparql_files_query(query=q_edam_format_3915_data_relations,
                                           filespath=['https://edamontology.org/EDAM_1.25.owl'],
                                           format='xml'
                                           )
print(sparql_resutls_to_prettytable(results=r_edam_format_3915_data_relations))


+-------------------------------------+--------------+-----------------------------------+------------------------+
|                format               | format_label |                data               |       data_label       |
+-------------------------------------+--------------+-----------------------------------+------------------------+
| http://edamontology.org/format_3915 |     Zarr     | http://edamontology.org/data_2535 |  Sequence tag profile  |
| http://edamontology.org/format_3915 |     Zarr     | http://edamontology.org/data_3112 | Gene expression matrix |
+-------------------------------------+--------------+-----------------------------------+------------------------+


In [20]:
# Output: new graph with simplified more direct edam:is_format_of relations

c_is_format_of = '''
CONSTRUCT {
    ?format edam:is_format_of ?data .
    }
WHERE {
        ?format rdfs:subClassOf+ edam:format_1915 ;
                rdfs:subClassOf ?restriction .
        ?restriction owl:onProperty edam:is_format_of ;
                     owl:someValuesFrom ?data .
}
'''

r_is_format_of = sparql_files_query(query=c_is_format_of, filespath=['https://edamontology.org/EDAM_1.25.owl'], format='xml' )
print(r_is_format_of.serialize(format='ttl').decode('utf-8'))

@prefix ns1: <http://edamontology.org/> .

ns1:format_1475 ns1:is_format_of ns1:data_0883,
        ns1:data_3870 .

ns1:format_1637 ns1:is_format_of ns1:data_1714 .

ns1:format_1638 ns1:is_format_of ns1:data_3110 .

ns1:format_1644 ns1:is_format_of ns1:data_3111 .

ns1:format_1919 ns1:is_format_of ns1:data_0849 .

ns1:format_1920 ns1:is_format_of ns1:data_1255 .

ns1:format_1921 ns1:is_format_of ns1:data_0863 .

ns1:format_2006 ns1:is_format_of ns1:data_0872 .

ns1:format_2013 ns1:is_format_of ns1:data_2600 .

ns1:format_2014 ns1:is_format_of ns1:data_0858 .

ns1:format_2017 ns1:is_format_of ns1:data_1501 .

ns1:format_2020 ns1:is_format_of ns1:data_0971 .

ns1:format_2021 ns1:is_format_of ns1:data_0972 .

ns1:format_2027 ns1:is_format_of ns1:data_2024 .

ns1:format_2030 ns1:is_format_of ns1:data_0962 .

ns1:format_2031 ns1:is_format_of ns1:data_0916 .

ns1:format_2035 ns1:is_format_of ns1:data_0846 .

ns1:format_2036 ns1:is_format_of ns1:data_0871 .

ns1:format_2037 ns1:is_format_of n

## EDAM: what links to other resources are contained in 

In EDAM links to other resources are established by property `rdfs:seeAlso`

The 2 cells bellow show the **links** from EDAM Format subclasses, follow by the link from Data types subclasses

Format:
* inconsistent links, from Wikipedia items to http://filext.com items

Data:
* Same as above,
* links to [Semanticscience Integrated Ontology](http://semanticscience.org/resource/) items, commonly subclasses of [information content entity](https://ontobee.org/ontology/SIO?iri=http://semanticscience.org/resource/SIO_000015), which are on very high level 


![SIO screenshot focused on information content entity](../media/SIO_010065.png)

In [25]:
q_edam_format_links = '''
SELECT ?format ?format_label ?link
WHERE {

        ?format rdfs:subClassOf+ edam:format_1915 ;
            rdfs:label ?format_label ;
            rdfs:seeAlso ?link .
}
ORDER BY ?format_label
'''

r_edam_format_links = sparql_files_query(query=q_edam_format_links,
                                           filespath=['https://edamontology.org/EDAM_1.25.owl'],
                                           format='xml'
                                           )
print(sparql_resutls_to_prettytable(results=r_edam_format_links))


+-------------------------------------+---------------------------------+--------------------------------------------------------------------------------+
|                format               |           format_label          |                                      link                                      |
+-------------------------------------+---------------------------------+--------------------------------------------------------------------------------+
| http://edamontology.org/format_3990 |               AVI               |              https://en.wikipedia.org/wiki/Audio_Video_Interleave              |
| http://edamontology.org/format_3982 |              CHAIN              |               https://genome.ucsc.edu/goldenPath/help/chain.html               |
| http://edamontology.org/format_3992 |           CIGAR format          |                    http://wiki.bits.vib.be/index.php/CIGAR/                    |
| http://edamontology.org/format_2200 |        FASTA-like (text)      

In [26]:

q_edam_data_links = '''
SELECT ?data ?data_label ?link
WHERE {

        ?data rdfs:subClassOf+ edam:data_0006 ;
            rdfs:label ?data_label ;
            rdfs:seeAlso ?link .
}
ORDER BY ?data_label
'''

r_edam_data_links = sparql_files_query(query=q_edam_data_links,
                                       filespath=['https://edamontology.org/EDAM_1.25.owl'],
                                       format='xml'
                                       )
print(sparql_resutls_to_prettytable(results=r_edam_data_links))


+-----------------------------------+-----------------------------+-------------------------------------------------------------------+
|                data               |          data_label         |                                link                               |
+-----------------------------------+-----------------------------+-------------------------------------------------------------------+
| http://edamontology.org/data_2091 |          Accession          |           http://semanticscience.org/resource/SIO_000675          |
| http://edamontology.org/data_2091 |          Accession          |           http://semanticscience.org/resource/SIO_000731          |
| http://edamontology.org/data_0842 |          Identifier         |            "http://purl.org/dc/elements/1.1/identifier"           |
| http://edamontology.org/data_0842 |          Identifier         |           http://semanticscience.org/resource/SIO_000115          |
| http://edamontology.org/data_0842 |          I